In [ ]:
import os 
os.chdir("..")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import SplineTransformer
from sklearn.linear_model import Ridge
import statsmodels.api as sm
from itertools import product
import pandas as pd
import patsy
from patsy import dmatrix
from st_repl import SpatialReg

sr = SpatialReg()


In [ ]:
gdf = sr.quasi_panel(alpha=5, beta=6,sigma=3, rho=0.7, seed=787)
gdf

In [ ]:
import patsy
from sklearn.linear_model import Ridge

formula = "y_true ~ total_employment + te(cr(lat, df=6), cr(lon, df=6), constraints='center')"

# 1. Generate design matrices
y, design = patsy.dmatrices(formula, gdf)

# 2. Fit the Ridge model
model = Ridge(alpha=0.01)
model.fit(design, y)

# 3. Predict the trend/spatial surface
gdf['y_tensor'] = model.predict(design)

# 4. Detrend by subtracting the model fit (residuals)
gdf['y_detrended'] = gdf['y_true'] - gdf['y_tensor']

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(gdf["y_true"], gdf['y_tensor'], 'o', label='Data Points')

In [ ]:

column_names = design.design_info.column_names

coef = model.coef_

coef_dict = dict(zip(column_names, coef))

X_vars = ['total_employment']
X_coefs = {var: coef_dict[var] for var in X_vars}

intercept = model.intercept_

print("Intercept:", intercept)
print("Linear coefficients:")
for var, val in X_coefs.items():
    print(f"  {var}: {val:.4f}")



In [ ]:
print(((gdf["y_true"] - gdf["y_tensor"])**2).sum()/ len(gdf))
plt.scatter(gdf["y_true"], gdf["y_tensor"]);

In [ ]:
xb = gdf[["total_employment","w_rook"]].values.reshape(-1,2)
y_d = gdf["y_true"].values.reshape(-1,1)
X = sm.add_constant(xb)
results = sm.OLS(y_d, X).fit()
print(results.summary())

In [ ]:
gdf["ols_rook"] = results.predict(X)

In [ ]:
print(((gdf["y_true"] - gdf["ols_rook"])**2).sum()/ len(gdf))
plt.scatter(gdf["y_true"], gdf["ols_rook"]);

In [ ]:
xb = gdf[["total_employment","w_queen"]].values.reshape(-1,2)
y_d = gdf["y_true"].values.reshape(-1,1)
X = sm.add_constant(xb)
results = sm.OLS(y_d, X).fit()
print(results.summary())

gdf["ols_queen"] = results.predict(X)

In [ ]:
print(((gdf["y_true"] - gdf["ols_queen"])**2).sum()/ len(gdf))
plt.scatter(gdf["y_true"], gdf["ols_queen"]);

In [ ]:
xb = gdf[["total_employment","w_knn6"]].values.reshape(-1,2)
y_d = gdf["y_true"].values.reshape(-1,1)
X = sm.add_constant(xb)
results = sm.OLS(y_d, X).fit()
print(results.summary())

gdf["ols_knn6"] = results.predict(X)

In [ ]:
print(((gdf["y_true"] - gdf["ols_knn6"])**2).sum()/ len(gdf))
plt.scatter(gdf["y_true"], gdf["ols_knn6"]);